# Graph Theory Fundamentals

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 07.01 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/07_Discrete_Mathematics/02_graph_theory_fundamentals.ipynb)

---

## 🎯 Learning Objective

Represent graphs with adjacency matrices, and derive the graph Laplacian as the foundation of spectral graph theory.

---

## 📐 Theory

A **graph** $G=(V,E)$ is nothing more than a set of **vertices** (or nodes) $V$ and a set of
**edges** $E$ connecting pairs of them. That's a deliberately minimal definition — its power is
how much structure (social networks, molecules, dependency chains, token relationships) fits
into "a bunch of things and some pairwise connections between them."

### Directed vs. undirected, weighted vs. unweighted

An edge $(u,v)$ is **undirected** if the connection is symmetric ("$u$ and $v$ are connected,"
no direction implied) or **directed** if it points one way ($u\to v$ doesn't imply $v\to u$).
Edges can also carry a **weight** $w(u,v)$ — a strength or cost rather than a bare yes/no
connection. A **complete graph** connects every vertex to every other vertex; with $n$
vertices, an undirected complete graph has $\binom{n}{2}$ edges (recall `07.01`'s combination
count — choosing which pair of vertices gets an edge is exactly a size-2 combination), while a
directed complete graph (every ordered pair connected) has $n(n-1)$ edges, or $n^2$ if
self-loops $(v,v)$ are allowed.

### The adjacency matrix

The **adjacency matrix** $A$ represents a graph on $n$ vertices as an $n\times n$ matrix where
$A_{ij}$ is the weight of the edge from vertex $i$ to vertex $j$ (or $1$/$0$ for
unweighted graphs, or $0$ if no edge exists). This single object converts graph theory into
linear algebra: an undirected graph gives a **symmetric** matrix ($A=A^\top$, since the edge
$i\to j$ and $j\to i$ carry the same weight), while a directed graph can give any square
matrix. Matrix powers count paths: $(A^k)_{ij}$ is the number of length-$k$ walks from $i$ to
$j$, since matrix multiplication sums over every intermediate vertex a walk could pass through
— exactly the same "sum over intermediate index" pattern as ordinary matrix multiplication
(`01.02`).

### Degree and the graph Laplacian

A vertex's **degree** is its total edge weight: $d_i = \sum_j A_{ij}$ (row sum). Collecting
degrees into a diagonal matrix $D=\text{diag}(d_1,\dots,d_n)$, the **graph Laplacian** is

$$L = D - A.$$

$L$ is always symmetric positive-semi-definite when $A$ is symmetric (recall
positive-definiteness from `01.06`), and it measures **smoothness**: for any signal
$\mathbf{f}\in\mathbb{R}^n$ assigning one number per vertex,

$$\mathbf{f}^\top L\,\mathbf{f} = \sum_{(i,j)\in E} w(i,j)\,(f_i-f_j)^2 \ge 0,$$

which is small exactly when $\mathbf{f}$ assigns similar values to vertices connected by
strong edges, and large when strongly-connected vertices disagree. This single identity is why
$L$ shows up everywhere signals live on graphs.

### Spectral graph theory

**Spectral graph theory** studies $L$'s eigenvalues $0=\lambda_1\le\lambda_2\le\cdots\le\lambda_n$
and eigenvectors. Three facts anchor the whole subject:

- $\lambda_1=0$ always, with eigenvector $\mathbf{1}=(1,\dots,1)$ — plugging a constant signal
  into $\mathbf{f}^\top L\mathbf{f}$ gives zero, since every term $(f_i-f_j)^2$ vanishes.
- The **multiplicity** of eigenvalue $0$ (how many independent eigenvectors achieve it) equals
  the number of **connected components** of the graph — each disconnected piece contributes its
  own constant-signal direction with zero smoothness cost.
- $\lambda_2$, the **algebraic connectivity** (or Fiedler value), measures how well-connected
  the graph is as a whole; its eigenvector (the **Fiedler vector**) assigns similar signs to
  vertices in the same loosely-defined cluster, which is exactly what **spectral clustering**
  exploits — cutting a graph along the sign change of the Fiedral vector tends to cut relatively
  few, relatively weak edges.

### Graphs as the natural structure for attention

Every one of these ideas has a direct target in modern deep learning: any set of $n$ items with
pairwise relationships *is* a graph, whether or not anyone draws it that way. Self-attention
(`01.10`) computes exactly such a pairwise relationship — a score between every pair of
tokens — which makes the attention matrix, read as an adjacency matrix, a literal (weighted,
directed, complete) graph over the sequence. We make this identification precise below.

---

In [ ]:
# Setup
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Drawing a small graph next to its adjacency matrix makes the two representations feel like the
same object viewed two ways. Comparing a connected graph's Laplacian eigenvalues against a
disconnected one then makes the "zero eigenvalue multiplicity = number of components" theorem
concrete rather than abstract.

In [ ]:
# A small weighted graph, its adjacency matrix, and the "zero eigenvalue count = number of
# connected components" theorem made concrete by comparing a connected vs. disconnected graph.
G = nx.Graph()
G.add_weighted_edges_from([(0, 1, 2), (0, 2, 1), (1, 2, 3), (2, 3, 1), (3, 4, 2), (2, 4, 1)])
A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
pos = nx.spring_layout(G, seed=1)
nx.draw(G, pos, ax=ax, with_labels=True, labels={i: f"v{i}" for i in G.nodes()},
         node_color="#4C72B0", font_color="white", node_size=700)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "weight"), ax=ax)
ax.set_title("A small weighted graph G")

ax2 = axes[1]
ax2.imshow(A, cmap="viridis")
n_nodes = len(G.nodes())
ax2.set_xticks(range(n_nodes)); ax2.set_yticks(range(n_nodes))
ax2.set_xticklabels([f"v{i}" for i in sorted(G.nodes())])
ax2.set_yticklabels([f"v{i}" for i in sorted(G.nodes())])
for i in range(n_nodes):
    for j in range(n_nodes):
        ax2.text(j, i, f"{A[i, j]:.0f}", ha="center", va="center",
                  color="white" if A[i, j] > 1.5 else "black")
ax2.set_title("Its adjacency matrix A\n(same information, matrix form)")

# Connected vs. disconnected: same number of vertices, compare Laplacian spectra directly.
G_disconnected = nx.Graph()
G_disconnected.add_edges_from([(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)])
eig_connected = np.linalg.eigvalsh(nx.laplacian_matrix(G).toarray().astype(float))
eig_disconnected = np.linalg.eigvalsh(nx.laplacian_matrix(G_disconnected).toarray().astype(float))

ax3 = axes[2]
ax3.plot(eig_connected, "o-", color="#4C72B0",
          label=f"connected graph ({nx.number_connected_components(G)} component)")
ax3.plot(eig_disconnected, "s-", color="#C44E52",
          label=f"disconnected graph ({nx.number_connected_components(G_disconnected)} components)")
ax3.axhline(0, color="gray", lw=0.5)
ax3.set_xlabel("eigenvalue index (sorted ascending)"); ax3.set_ylabel("Laplacian eigenvalue")
ax3.set_title("# of (near-)zero eigenvalues =\n# of connected components")
ax3.legend()
plt.tight_layout()
plt.show()

print(f"Connected graph eigenvalues:    {np.round(eig_connected, 4)}")
print(f"Disconnected graph eigenvalues: {np.round(eig_disconnected, 4)}")
print(f"Zero eigenvalues (connected):    {np.sum(np.abs(eig_connected) < 1e-9)} "
      f"(matches {nx.number_connected_components(G)} component)")
print(f"Zero eigenvalues (disconnected): {np.sum(np.abs(eig_disconnected) < 1e-9)} "
      f"(matches {nx.number_connected_components(G_disconnected)} components)")

## 🐍 Implementation from Scratch

We build the adjacency matrix, degree matrix, and Laplacian from a plain edge list (no
`networkx` graph object required for the math itself), verify that matrix powers of $A$ count
walks by cross-checking against brute-force recursive walk-counting, and confirm the
Laplacian's smoothness identity and its "zero eigenvalues = connected components" theorem
numerically.

In [ ]:
def build_adjacency(n_vertices, edges, weighted=False):
    """Adjacency matrix from a plain edge list -- the ground-truth representation
    every other graph computation here is built on top of."""
    A = np.zeros((n_vertices, n_vertices))
    for edge in edges:
        i, j, w = (edge[0], edge[1], edge[2]) if weighted else (edge[0], edge[1], 1.0)
        A[i, j] = w
        A[j, i] = w  # undirected: the connection is symmetric
    return A

def graph_laplacian(A):
    """L = D - A, where D is the diagonal degree matrix (row sums of A)."""
    degrees = A.sum(axis=1)
    return np.diag(degrees) - A

# --- Path counting: does A^k really count length-k walks? Cross-check vs brute-force recursion ---
edges_small = [(0, 1), (1, 2), (2, 3), (3, 0), (1, 3)]
A_small = build_adjacency(4, edges_small)

def count_walks_bruteforce(A, i, j, k):
    """Independent, slow-but-obviously-correct walk counter: recurse one hop at a time."""
    if k == 0:
        return 1.0 if i == j else 0.0
    return sum(A[i, mid] * count_walks_bruteforce(A, mid, j, k - 1) for mid in range(len(A)))

A_squared = A_small @ A_small
all_match = all(
    A_squared[i, j] == count_walks_bruteforce(A_small, i, j, 2)
    for i in range(4) for j in range(4)
)
print(f"A^2 matches brute-force walk counting for every (i,j) pair: {all_match}")
print(f"e.g. A^2[0,3] = {A_squared[0, 3]:.0f} length-2 walks from vertex 0 to vertex 3")

# --- Laplacian smoothness identity: f^T L f == sum over edges of w_ij*(f_i-f_j)^2 ---
rng = np.random.default_rng(2)
L_small = graph_laplacian(A_small)
f = rng.normal(size=4)
quadratic_form = f @ L_small @ f
edge_sum = sum(A_small[i, j] * (f[i] - f[j]) ** 2 for i, j in edges_small)
print(f"\nf^T L f = {quadratic_form:.6f}, sum_edges w_ij*(f_i-f_j)^2 = {edge_sum:.6f} "
      f"(match: {np.isclose(quadratic_form, edge_sum)})")

# --- Zero-eigenvalue multiplicity == number of connected components ---
def n_connected_components(A):
    """Simple flood-fill component counter -- independent of the eigenvalue-based claim we're
    about to check, so it acts as ground truth rather than assuming the theorem it verifies."""
    n = len(A)
    visited = np.zeros(n, dtype=bool)
    components = 0
    for start in range(n):
        if visited[start]:
            continue
        components += 1
        stack = [start]
        while stack:
            node = stack.pop()
            if visited[node]:
                continue
            visited[node] = True
            neighbors = np.nonzero(A[node])[0]
            stack.extend(neighbors)
    return components

for name, edges, n_v in [
    ("connected (4-cycle+chord)", edges_small, 4),
    ("disconnected (two triangles)", [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)], 6),
]:
    A_test = build_adjacency(n_v, edges)
    L_test = graph_laplacian(A_test)
    eigvals = np.linalg.eigvalsh(L_test)
    n_zero_eigs = np.sum(np.abs(eigvals) < 1e-9)
    n_components = n_connected_components(A_test)
    print(f"\n{name}: {n_zero_eigs} near-zero eigenvalues, "
          f"{n_components} connected components (match: {n_zero_eigs == n_components})")

## 🤖 Why This Matters for AI

Self-attention (`01.10`, [Linear Algebra in Transformers](../01_Linear_Algebra/10_linear_algebra_in_transformers.ipynb))
computes $\text{softmax}(QK^\top/\sqrt{d_k})$ — a row-normalized matrix of pairwise scores
between every pair of tokens. That is *literally* a weighted adjacency matrix: entry $(i,j)$ is
the weight of a directed edge from token $i$ to token $j$, and because every token attends to
every other token (and to itself), the underlying graph is a **complete directed graph** on the
sequence. **Graph Neural Networks (GNNs)** generalize this same idea to graphs that *aren't*
complete — a GNN layer aggregates each node's neighbors' features weighted by learned
edge/attention weights, exactly like one attention head aggregates value vectors, but restricted
to whatever edges the input graph (a molecule, a social network, a knowledge graph) actually
has. Below, we build a real attention matrix from Q/K vectors, treat it as a graph adjacency
matrix, draw it with `networkx`, and compute its Laplacian spectrum.

In [ ]:
# Build a REAL attention matrix from Q/K vectors (not a toy stand-in), then treat it as a
# graph: visualize with networkx and compute its Laplacian spectrum.
rng = np.random.default_rng(11)  # fresh, self-contained RNG so this cell doesn't depend on
                                   # random state consumed by earlier cells (see Theory/Impl above)

tokens = ["The", "cat", "sat", "on", "mat"]
n_tokens, d_k = len(tokens), 8

X = rng.normal(scale=1.0, size=(n_tokens, d_k))     # toy token embeddings
W_q = rng.normal(scale=0.7, size=(d_k, d_k))
W_k = rng.normal(scale=0.7, size=(d_k, d_k))
Q, K = X @ W_q, X @ W_k

def softmax_rows(x):
    x = x - x.max(axis=-1, keepdims=True)  # numerically stable softmax (recall 06.02)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

attention = softmax_rows(Q @ K.T / np.sqrt(d_k))   # THIS is the adjacency matrix
print("Attention matrix (row i = how much token i attends to each token j):")
print(np.round(attention, 3))
print(f"Row sums (softmax property -> out-degree of every node is exactly 1): "
      f"{np.round(attention.sum(axis=1), 6)}")

# Treat `attention` as the weighted adjacency matrix of a directed complete graph and draw it.
G_attn = nx.DiGraph()
for i, tok in enumerate(tokens):
    G_attn.add_node(i, label=tok)
for i in range(n_tokens):
    for j in range(n_tokens):
        if i != j and attention[i, j] > 0.05:   # only draw edges strong enough to see clearly
            G_attn.add_edge(i, j, weight=attention[i, j])

fig, ax = plt.subplots(figsize=(7, 6))
pos = nx.circular_layout(G_attn)
weights = [G_attn[u][v]["weight"] * 6 for u, v in G_attn.edges()]
nx.draw_networkx_nodes(G_attn, pos, node_color="#4C72B0", node_size=1400, ax=ax)
nx.draw_networkx_labels(G_attn, pos, labels={i: t for i, t in enumerate(tokens)},
                          font_color="white", ax=ax)
nx.draw_networkx_edges(G_attn, pos, width=weights, alpha=0.6, arrowsize=15,
                         connectionstyle="arc3,rad=0.15", ax=ax)
ax.set_title("Self-attention IS a weighted directed complete graph\n(edges below weight 0.05 hidden for clarity)")
ax.axis("off")
plt.tight_layout()
plt.show()

# The graph Laplacian and its spectrum -- symmetrize first, since attention (unlike an
# undirected graph's adjacency matrix) is not symmetric: token i attending strongly to j
# doesn't mean j attends equally strongly back.
attention_sym = (attention + attention.T) / 2
degrees_sym = np.diag(attention_sym.sum(axis=1))
laplacian = degrees_sym - attention_sym
eigvals, eigvecs = np.linalg.eigh(laplacian)

print(f"\nSymmetrized attention Laplacian eigenvalues: {np.round(eigvals, 4)}")
print(f"Smallest eigenvalue ~0 (confirms the token graph is fully connected): {eigvals[0]:.2e}")
print(f"Number of near-zero eigenvalues: {np.sum(np.abs(eigvals) < 1e-9)} "
      "(1 connected component, matching a complete graph where every token attends to every other)")

# Rayleigh-quotient sanity check from the Theory section: the eigenvector belonging to the
# LARGEST eigenvalue should be the least "smooth" signal on this token graph.
rng_check = np.random.default_rng(99)
f_random = rng_check.normal(size=n_tokens)
print(f"\nSmoothness f^T L f: smoothest eigvec={eigvecs[:, 0] @ laplacian @ eigvecs[:, 0]:.2e}, "
      f"random signal={f_random @ laplacian @ f_random:.3f}, "
      f"roughest eigvec={eigvecs[:, -1] @ laplacian @ eigvecs[:, -1]:.4f} "
      f"(should equal largest eigenvalue {eigvals[-1]:.4f})")

## 🏋️ Exercises

### Warm-up
1. By hand, write the adjacency matrix for a 4-vertex undirected "path graph" $0-1-2-3$ (no
   other edges). Compute its degree matrix $D$ and Laplacian $L=D-A$, then verify with
   `build_adjacency`/`graph_laplacian` that your hand computation matches.

### Practice
2. For the path graph from Exercise 1, compute the Laplacian's eigenvalues with
   `np.linalg.eigvalsh`. Confirm exactly one eigenvalue is (numerically) zero, matching the
   graph's single connected component. Then remove the edge $(1,2)$ and recompute — how many
   zero eigenvalues appear now, and does that match the new number of components?

### Challenge
3. In the AI section, we thresholded attention edges at weight $>0.05$ purely for drawing
   clarity, but computed the Laplacian from the *full*, unthresholded attention matrix. Redo
   the Laplacian computation using only the thresholded (edges below 0.05 zeroed out) attention
   matrix instead, symmetrizing and renormalizing rows to sum to 1 first. How much do the
   eigenvalues change? Explain what this tells you about **sparse attention** mechanisms (used
   in models like Longformer and BigBird), which deliberately restrict attention to a subset of
   token pairs rather than the full complete graph, in terms of a tradeoff between compute cost
   and how much of the graph's spectral structure survives.

---

## 📚 Further Reading
- Chung, *Spectral Graph Theory*, Ch. 1 (the Laplacian and its eigenvalues)
- Veličković et al., ["Graph Attention Networks"](https://arxiv.org/abs/1710.10903)
- Beltagy, Peters & Cohan, ["Longformer: The Long-Document Transformer"](https://arxiv.org/abs/2004.05150) (sparse attention as a sparser graph)

---

*Next notebook: [Trees and Search](03_trees_and_search.ipynb)*